In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import os

print("All libraries loaded successfully ✓")
print(f"pandas: {pd.__version__}")
print(f"yfinance: {yf.__version__}")

All libraries loaded successfully ✓
pandas: 3.0.1
yfinance: 1.2.0


In [3]:
# Asset list
TICKERS = ["GLD", "SPY", "BBCA.JK", "TLKM.JK"]
START = "2021-01-01"
END   = "2025-12-31"

# Download stocks and gold
print("Downloading market data...")
data = yf.download(TICKERS, start=START, end=END, auto_adjust=True)["Close"]

# Download BTC separately to avoid database lock
print("Downloading Bitcoin...")
btc = yf.download("BTC-USD", start=START, end=END, auto_adjust=True)["Close"]
btc.columns = ["BTC-USD"]

# Combine into one dataframe
raw = pd.concat([data, btc], axis=1)

# Rename columns to plain names
raw.columns = ["Gold", "SP500", "BCA", "Telkom", "Bitcoin"]

# Basic info
print("Shape:", raw.shape)
print("Date range:", raw.index[0].date(), "to", raw.index[-1].date())
print("Missing values per asset:")
print(raw.isnull().sum())
raw.head()

[*********************100%***********************]  4 of 4 completed
[*********************100%***********************]  1 of 1 completed
C:\Users\Windows 10\AppData\Local\Temp\ipykernel_7896\4031667304.py:16: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  raw = pd.concat([data, btc], axis=1)


Shape: (1825, 5)
Date range: 2021-01-01 to 2025-12-30
Missing values per asset:
Gold       620
SP500      571
BCA        571
Telkom     620
Bitcoin      0
dtype: int64


,Gold,SP500,BCA,Telkom,Bitcoin
Date,,,,,
2021-01-01,NaN,NaN,NaN,NaN,29374.152344
2021-01-02,NaN,NaN,NaN,NaN,32127.267578
2021-01-03,NaN,NaN,NaN,NaN,32782.023438
2021-01-04,5775.54248,182.330002,343.319092,2687.039062,31971.914062
2021-01-05,5991.01709,182.869995,345.683685,2671.640869,33992.429688


In [14]:
# Fix the concat warning by explicitly setting sort=True
raw = pd.concat([data, btc], axis=1, sort=True)
raw.columns = ["Gold", "SP500", "BCA", "Telkom", "Bitcoin"]

# Drop rows where ALL stock columns are missing (weekends and holidays)
raw = raw.dropna(subset=["SP500", "BCA"], how="all")

# Forward fill remaining small gaps (public holidays where one market closed but not another)
raw = raw.ffill()

# Confirm result
print("Shape after cleaning:", raw.shape)
print("Date range:", raw.index[0].date(), "to", raw.index[-1].date())
print("Missing values after cleaning:")
print(raw.isnull().sum())
raw.head(10)

Shape after cleaning: (1254, 5)
Date range: 2021-01-04 to 2025-12-30
Missing values after cleaning:
Gold       0
SP500      0
BCA        0
Telkom     0
Bitcoin    0
dtype: int64


,Gold,SP500,BCA,Telkom,Bitcoin
Date,,,,,
2021-01-04,6028.374512,182.330002,343.319122,2687.039307,31971.914062
2021-01-05,6253.280273,182.869995,345.683685,2671.640869,33992.429688
2021-01-06,6125.393555,179.899994,347.750427,2594.647949,36824.363281
2021-01-07,6143.032715,179.479996,352.917084,2610.046631,39371.042969
2021-01-08,6218.001953,173.339996,354.927948,2748.633301,40797.609375
2021-01-11,6478.187500,173.000000,352.535400,2771.730713,35566.656250
2021-01-12,6315.020020,174.119995,352.609802,2702.437744,33922.960938
2021-01-13,6279.741211,173.369995,353.559357,2679.339844,37316.359375
2021-01-14,6191.541504,173.279999,352.321259,2694.738281,39187.328125


In [15]:
import os

# Create data folder if it does not exist
os.makedirs("../data", exist_ok=True)

# Save cleaned data to CSV
raw.to_csv("../data/prices_clean.csv")

print("File saved to data/prices_clean.csv")
print("Shape:", raw.shape)

File saved to data/prices_clean.csv
Shape: (1254, 5)
